# AMPS II · Week 06 — HW6
## Newmark stability and accuracy, slenderness, buckling studies, and an LLM-generated Newmark script

**Due before Week 7 · 100 points.** Submit the executed notebook: *Runtime → Restart session and run all*, fill in the AI-usage log at the end.
Every number you report must come from a cell of this notebook. The first cell of Q4 runs an LLM-generated script that is **meant** to be wrong.

| Q | Topic | Points |
|---|---|:---:|
| 1 | Newmark stability and period error, Rayleigh damping, the buckling eigenproblem (theory) | 20 |
| 2 | Dynamics: the explicit stability limit, a numerically damped scheme, slenderness and Euler–Bernoulli | 30 |
| 3 | Buckling: element order and mesh, slenderness and shear, which $K_G$ | 30 |
| 4 | Debugging an LLM-generated Newmark script | 20 |

## ⚙️ Setup — run once per runtime
Installs DOLFINx 0.11 and registers the `%%fenicsx` cell magic. Re-run after every runtime restart.

In [ ]:
# ══════════════════════════════════════════════════════════════════
#  FEniCSx on Google Colab — run once per runtime (≈ 3–5 min)
#  https://github.com/SeoulTechPSE/fenicsx-colab
# ══════════════════════════════════════════════════════════════════
from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)

# OpenMPI 5.x slot patch: Colab has 2 CPUs; allow up to 4 ranks (oversubscribed)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['OMPI_MCA_osc'] = '^ucx'   # skip the UCX one-sided module (harmless start-up message)

DOLFINX_VERSION = '0.11'
USE_COMPLEX = False
USE_CLEAN   = False
opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

# Folder shared by %%fenicsx cells (computation) and ordinary cells (plotting)
os.makedirs('/tmp/fnx', exist_ok=True)

In [ ]:
%%fenicsx --info -np 2

The model modules from the lab (unchanged). Run the three cells once.

In [ ]:
%%writefile /tmp/fnx/sdof.py
"""The Newmark family on one oscillator  q'' + 2 zeta w q' + w^2 q = f  -- NumPy only.
One step maps (q, v, a)_n -> (q, v, a)_{n+1}; for f = 0 this is a 3 x 3 amplification matrix A(Omega = w dt).
Stability: spectral radius rho(A) <= 1. Accuracy: the angle of the complex eigenvalues of A vs Omega (period error)."""
import numpy as np

def amplification(Omega, beta=0.25, gamma=0.5, zeta=0.0):
    """3 x 3 amplification matrix of the Newmark step for q'' + 2 zeta w q' + w^2 q = 0, time in units of 1/w (w = 1)"""
    dt = Omega
    # implicit relation for a_{n+1}: a + 2 zeta (v_p + gamma dt a) + (q_p + beta dt^2 a) = 0
    den = 1 + 2 * zeta * gamma * dt + beta * dt**2
    A = np.zeros((3, 3))
    for j, e in enumerate(np.eye(3)):
        q, v, a = e
        qp = q + dt * v + dt**2 * (0.5 - beta) * a
        vp = v + dt * (1 - gamma) * a
        an = -(qp + 2 * zeta * vp) / den
        A[:, j] = [qp + beta * dt**2 * an, vp + gamma * dt * an, an]
    return A

def spectral_radius(Omega, **kw):
    return float(np.max(np.abs(np.linalg.eigvals(amplification(Omega, **kw)))))

def period_ratio(Omega, **kw):
    """numerical period / exact period for the undamped oscillator (from the phase of the principal eigenvalue)"""
    lam = np.linalg.eigvals(amplification(Omega, **kw))
    osc = lam[lam.imag > 1e-14]                     # the complex pair of the oscillation (the third eigenvalue is spurious)
    return Omega / np.angle(osc[0]) if len(osc) else np.inf

def integrate(w, dt, n, f=1.0, q0=0.0, v0=0.0, beta=0.25, gamma=0.5, zeta=0.0):
    """step response (constant f from t = 0+) with the consistent initial acceleration; returns t, q"""
    q, v = q0, v0
    a = f - 2 * zeta * w * v - w**2 * q
    out = [q]
    for _ in range(n):
        qp = q + dt * v + dt**2 * (0.5 - beta) * a
        vp = v + dt * (1 - gamma) * a
        a = (f - 2 * zeta * w * vp - w**2 * qp) / (1 + 2 * zeta * w * gamma * dt + w**2 * beta * dt**2)
        q, v = qp + beta * dt**2 * a, vp + gamma * dt * a
        out.append(q)
    return np.arange(n + 1) * dt, np.array(out)

def step_exact(w, t, f=1.0, zeta=0.0):
    """exact step response of the (damped) oscillator, q(0) = q'(0) = 0"""
    qs = f / w**2
    wd = w * np.sqrt(1 - zeta**2)
    return qs * (1 - np.exp(-zeta * w * t) * (np.cos(wd * t) + zeta / np.sqrt(1 - zeta**2) * np.sin(wd * t)))

In [ ]:
%%writefile /tmp/fnx/dyn.py
"""Elastodynamics of a 2D cantilever (plane stress, thickness 1) in DOLFINx 0.11 + SLEPc:
mass and stiffness matrices, natural frequencies (shift-invert), static solution, Newmark time integration with Rayleigh damping
(PETSc vectors, the effective matrix factorised ONCE), and -- serially, for small meshes -- the exact semi-discrete solution by
complete modal superposition (dense matrices)."""
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from slepc4py import SLEPc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

BETA_L = [1.875104069, 4.694091133, 7.854757438, 10.99554073]       # cantilever: beta_n L of Euler-Bernoulli

def lu(ksp):
    ksp.setType("preonly"); ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    return ksp

class Cantilever:
    """[0, L] x [0, H], clamped at x = 0, a shear traction of total P (per unit thickness) on x = L, downwards.
    plane = "stress" (a thin beam: lambda* = E nu / (1 - nu^2)) or "strain"."""
    def __init__(self, L, H, nx, ny, E, nu, rho, P=1.0, plane="stress", deg=2):
        self.L, self.H, self.E, self.nu, self.rho, self.P = L, H, E, nu, rho, P
        msh = self.msh = mesh.create_rectangle(comm, [[0, 0], [L, H]], [nx, ny], cell_type=mesh.CellType.triangle)
        mu = E / (2 * (1 + nu)); lm = E * nu / (1 - nu**2) if plane == "stress" else E * nu / ((1 + nu) * (1 - 2 * nu))
        self.mu = mu
        V = self.V = fem.functionspace(msh, ("Lagrange", deg, (2,)))
        u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        eps = lambda w: ufl.sym(ufl.grad(w))
        self.sig = lambda w: lm * ufl.tr(eps(w)) * ufl.Identity(2) + 2 * mu * eps(w)
        left = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], 0))
        right = np.sort(mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], L)))
        self.ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 1, right, np.ones(len(right), dtype=np.int32)))
        self.bc = fem.dirichletbc(np.zeros(2), fem.locate_dofs_topological(V, 1, left), V)
        self.mf = fem.form(rho * ufl.inner(u, v) * ufl.dx)
        self.kf = fem.form(ufl.inner(self.sig(u), eps(v)) * ufl.dx)
        self.K = assemble_matrix(self.kf, bcs=[self.bc], diag=1.0); self.K.assemble()     # BC rows: identity
        self.M = assemble_matrix(self.mf, bcs=[self.bc], diag=0.0); self.M.assemble()     # BC rows: zero -> BC "modes" at infinity
        self.M1 = assemble_matrix(self.mf, bcs=[self.bc], diag=1.0); self.M1.assemble()   # BC rows: identity (time stepping, a0)
        self.f = assemble_vector(fem.form(ufl.dot(fem.Constant(msh, np.array([0.0, -P / H])), v) * self.ds(1)))
        apply_lifting(self.f, [self.kf], [[self.bc]]); self.f.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(self.f, [self.bc])
        self.g = assemble_vector(fem.form(v[1] / H * self.ds(1)))                       # tip = mean u_y on x = L, as a linear functional
        self.g.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(self.g, [self.bc])
        self.Ix, self.A = H**3 / 12, H

    def tip(self, x):
        """mean u_y on x = L of a PETSc vector of displacements"""
        return self.g.dot(x)

    def euler_bernoulli(self, n):
        return BETA_L[n - 1]**2 * np.sqrt(self.E * self.Ix / (self.rho * self.A * self.L**4))

    def static(self):
        u = fem.Function(self.V); k = lu(PETSc.KSP().create(comm)); k.setOperators(self.K)
        k.solve(self.f, u.x.petsc_vec); u.x.scatter_forward()
        return u

    def modes(self, n=6):
        """the n lowest natural frequencies (rad/s), M-normalised mode vectors, and the share of axial motion of each mode"""
        e = SLEPc.EPS().create(comm); e.setOperators(self.K, self.M); e.setProblemType(SLEPc.EPS.ProblemType.GHEP)
        e.setDimensions(n); e.setTarget(0.0); e.setWhichEigenpairs(SLEPc.EPS.Which.TARGET_MAGNITUDE)
        st = e.getST(); st.setType("sinvert"); lu(st.getKSP()); e.setTolerances(1e-10, 500); e.solve()
        out = []
        for i in range(e.getConverged()):
            x = self.K.createVecRight(); lam = e.getEigenpair(i, x).real
            Mx = self.M.createVecRight(); self.M.mult(x, Mx); x.scale(1 / np.sqrt(x.dot(Mx)))
            ux, uy = x.array_r[0::2], x.array_r[1::2]
            ax = comm.allreduce(float(ux @ ux), op=MPI.SUM); tot = ax + comm.allreduce(float(uy @ uy), op=MPI.SUM)
            out.append((np.sqrt(lam), x, ax / tot))
        out.sort(key=lambda r: r[0])
        return out[:n]

    def newmark(self, dt, nsteps, beta=0.25, gamma=0.5, aM=0.0, aK=0.0, load=1.0, u0=None, record=()):
        """M a + C v + K u = load * f,  C = aM M + aK K.  Starts from u0 (PETSc vector or None = 0), v = 0 and the CONSISTENT
        acceleration a0 = M^-1 (load f - K u0). The matrix M + gamma dt C + beta dt^2 K is assembled and factorised once.
        record: extra vectors y; returns a dict with t, tip, energy (kinetic + strain - load work) and y . (M u) for each y."""
        M, K, f = self.M, self.K, self.f
        A = self.M1.copy(); A.axpy(gamma * dt * aM, M); A.axpy(gamma * dt * aK + beta * dt**2, K)   # BC rows: identity-dominated (also for beta = 0)
        ksp = lu(PETSc.KSP().create(comm)); ksp.setOperators(A)
        u, v, a, up, vp, r, tmp = (f.duplicate() for _ in range(7))
        u.set(0.0); v.set(0.0)
        if u0 is not None:
            u0.copy(u)
        r = f.copy(); r.scale(load); K.mult(u, tmp); r.axpy(-1.0, tmp); set_bc(r, [self.bc], alpha=0.0)
        k0 = lu(PETSc.KSP().create(comm)); k0.setOperators(self.M1); k0.solve(r, a)
        hist = {"t": [0.0], "tip": [self.tip(u)], "energy": [self._energy(u, v, load)], "rec": [[y.dot(self._Mx(u)) for y in record]]}
        for n in range(1, nsteps + 1):
            u.copy(up); up.axpy(dt, v); up.axpy(dt**2 * (0.5 - beta), a)
            v.copy(vp); vp.axpy(dt * (1 - gamma), a)
            r = f.copy(); r.scale(load)
            K.mult(up, tmp); r.axpy(-1.0, tmp)
            if aM or aK:
                M.mult(vp, tmp); r.axpy(-aM, tmp); K.mult(vp, tmp); r.axpy(-aK, tmp)
            set_bc(r, [self.bc], alpha=0.0)
            ksp.solve(r, a)
            up.copy(u); u.axpy(beta * dt**2, a)
            vp.copy(v); v.axpy(gamma * dt, a)
            hist["t"].append(n * dt); hist["tip"].append(self.tip(u)); hist["energy"].append(self._energy(u, v, load))
            hist["rec"].append([y.dot(self._Mx(u)) for y in record])
        return {k: np.array(x) for k, x in hist.items()}

    def _Mx(self, x):
        y = x.duplicate(); self.M.mult(x, y); return y

    def _energy(self, u, v, load):
        Kv = u.duplicate(); self.K.mult(u, Kv)
        return 0.5 * v.dot(self._Mx(v)) + 0.5 * u.dot(Kv) - load * self.f.dot(u)

    def dense_modal(self):
        """SERIAL only: all modes from the dense matrices (BC rows removed). Returns omega, the tip and load projections, so that
        the exact semi-discrete step response is  tip(t) = sum_i g_i c_i / omega_i^2 (1 - cos omega_i t)."""
        from scipy.linalg import eigh
        assert comm.size == 1, "dense_modal needs -np 1"
        Kd, Md = PETSc.Mat(), PETSc.Mat()
        self.K.convert("dense", Kd); self.M.convert("dense", Md)                         # into NEW matrices: convert() without out= is in place
        Kd, Md = Kd.getDenseArray(), Md.getDenseArray()
        free = np.setdiff1d(np.arange(Kd.shape[0]), self.bc.dof_indices()[0])
        lam, Phi = eigh(Kd[np.ix_(free, free)], Md[np.ix_(free, free)])
        return np.sqrt(lam), Phi.T @ self.g.array[free], Phi.T @ self.f.array[free]

In [ ]:
%%writefile /tmp/fnx/buckle.py
"""Linear buckling of a 3D column (hexahedra) in DOLFINx 0.11 + SLEPc: pre-buckling stress under a reference load, the geometric
stiffness  K_G = int sigma0 : (grad u . grad v^T),  and the eigenproblem  K phi = lambda (-K_G) phi,  solved as
(-K_G) phi = (1/lambda) K phi  with the positive definite K as the B matrix (no special treatment of the BC rows needed)."""
import numpy as np
import ufl
from mpi4py import MPI
from petsc4py import PETSc
from slepc4py import SLEPc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import apply_lifting, assemble_matrix, assemble_vector, set_bc

comm = MPI.COMM_WORLD
def say(*a):
    if comm.rank == 0:
        print(*a, flush=True)

def lu(ksp):
    ksp.setType("preonly"); ksp.getPC().setType("lu"); ksp.getPC().setFactorSolverType("mumps")
    return ksp

class Column:
    """[0, b] x [0, h] x [0, L], clamped at z = 0, a uniform compressive traction P_ref / (b h) on z = L (free end).
    Bending with deflection in y uses I = b h^3/12 (the weak axis if h < b), deflection in x uses I = h b^3/12."""
    def __init__(self, b, h, L, nx, ny, nz, E, nu, deg=2, kg="full"):
        self.b, self.h, self.L, self.E, self.nu = b, h, L, E, nu
        msh = self.msh = mesh.create_box(comm, [[0, 0, 0], [b, h, L]], [nx, ny, nz], mesh.CellType.hexahedron)
        mu, lm = E / (2 * (1 + nu)), E * nu / ((1 + nu) * (1 - 2 * nu)); self.G = mu
        V = self.V = fem.functionspace(msh, ("Lagrange", deg, (3,)))
        u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
        eps = lambda w: ufl.sym(ufl.grad(w))
        sig = lambda w: lm * ufl.tr(eps(w)) * ufl.Identity(3) + 2 * mu * eps(w)
        base = mesh.locate_entities_boundary(msh, 2, lambda x: np.isclose(x[2], 0))
        top = np.sort(mesh.locate_entities_boundary(msh, 2, lambda x: np.isclose(x[2], L)))
        ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 2, top, np.ones(len(top), dtype=np.int32)))
        self.bc = fem.dirichletbc(np.zeros(3), fem.locate_dofs_topological(V, 2, base), V)
        self.Pref = self.euler(1, "y")                                    # reference load: the first Euler load (lambda ~ 1)
        kf = fem.form(ufl.inner(sig(u), eps(v)) * ufl.dx)
        self.K = assemble_matrix(kf, bcs=[self.bc], diag=1.0); self.K.assemble()
        fv = assemble_vector(fem.form(ufl.dot(fem.Constant(msh, np.array([0.0, 0.0, -self.Pref / (b * h)])), v) * ds(1)))
        apply_lifting(fv, [kf], [[self.bc]]); fv.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(fv, [self.bc])
        self.u0 = fem.Function(V); k = lu(PETSc.KSP().create(comm)); k.setOperators(self.K); k.solve(fv, self.u0.x.petsc_vec)
        self.u0.x.scatter_forward()
        s0 = sig(self.u0)
        if kg == "full":
            gform = ufl.inner(s0, ufl.dot(ufl.grad(u), ufl.grad(v).T))                 # sigma_ij u_k,i v_k,j
        elif kg == "zz":
            gform = s0[2, 2] * ufl.dot(u.dx(2), v.dx(2))                               # sigma_zz u_k,z v_k,z (all k)
        else:
            gform = s0[2, 2] * u[2].dx(2) * v[2].dx(2)                                 # only k = z
        self.mKG = assemble_matrix(fem.form(-gform * ufl.dx), bcs=[self.bc], diag=0.0); self.mKG.assemble()
        self.ndofs = V.dofmap.index_map.size_global * 3

    def euler(self, n, axis):
        """Euler load of a clamped-free column, mode n, deflection along axis 'y' (I = b h^3/12) or 'x' (I = h b^3/12)"""
        I = self.b * self.h**3 / 12 if axis == "y" else self.h * self.b**3 / 12
        return (2 * n - 1)**2 * np.pi**2 * self.E * I / (4 * self.L**2)

    def solve(self, nev=6):
        """the nev smallest load factors lambda (P_cr = lambda P_ref) with their modes and the dominant deflection direction"""
        e = SLEPc.EPS().create(comm); e.setOperators(self.mKG, self.K); e.setProblemType(SLEPc.EPS.ProblemType.GHEP)
        e.setDimensions(nev); e.setWhichEigenpairs(SLEPc.EPS.Which.LARGEST_REAL)       # theta = 1/lambda: largest theta = smallest lambda
        lu(e.getST().getKSP()); e.setTolerances(1e-10, 1000); e.solve()
        out = []
        for i in range(e.getConverged()):
            x = self.K.createVecRight(); th = e.getEigenpair(i, x).real
            if th <= 0:
                continue
            a = x.array_r
            nx, ny = (comm.allreduce(float(a[c::3] @ a[c::3]), op=MPI.SUM) for c in (0, 1))
            out.append((1 / th, "y" if ny > nx else "x", x))
        out.sort(key=lambda r: r[0])
        return out[:nev], e.getConverged()

## Q1 — Theory (20 pts)

**(a) (8)** For $\ddot q+\omega^2q=0$ write one Newmark step as $\mathbf x_{n+1}=A\,\mathbf x_n$ with $\mathbf x=(q,\Delta t\dot q)$ (eliminate $\ddot q$). For average acceleration show
$\det A=1$ and $|\operatorname{tr}A|<2$ for every $\Omega=\omega\Delta t$ (so $|\lambda|=1$). For central difference ($\beta=0$) show that the scheme is stable iff $\Omega\le2$.

**(b) (6)** With $C=\alpha_MM+\alpha_KK$, derive $\zeta_i=\frac{\alpha_M}{2\omega_i}+\frac{\alpha_K\omega_i}2$. Choose $\alpha_M,\alpha_K$ for $\zeta=5\,\%$ at $\omega_1$ **and** $\omega_2$ of the lab cantilever
(523.59 and 3141.78 rad/s). Which damping does mode 4 (8278.64 rad/s) get?

**(c) (6)** For a column under a uniform axial stress $\sigma_{zz}=-P/A$ (all other $\sigma^0_{ij}=0$), reduce $K_G$ to the beam form $-P\int w'\hat w'\,dz$ for a lateral deflection $w(z)$.
Why does keeping only $k=z$ in $\sigma^0_{ij}u_{k,i}v_{k,j}$ lose buckling?

> *your answer*

## Q2 — Dynamics (30 pts)

The lab's cantilever ($20\times4$ P2, step load $P$).

**(a) (12)** Central difference ($\beta=0$, $\gamma=\tfrac12$) with the consistent mass. Get $\omega_{\max}$ from `dense_modal()` and run 4000 steps with $\Delta t=0.99\cdot2/\omega_{\max}$ and with
$1.01\cdot2/\omega_{\max}$. Report what happens, the time span 4000 steps cover in units of $T_1$, and what an explicit run to $2T_1$ would cost compared with average
acceleration at $T_1/\Delta t=100$.

In [ ]:
%%fenicsx -np 1
# TODO: Q2(a) — the explicit stability limit

> *your answer (Q2a)*

**(b) (8)** A numerically damped Newmark scheme ($\gamma=0.6$, $\beta=0.3025$) against average acceleration, for $T_1/\Delta t$ = 25, 50, 100: the mode-1 error and the tip error (against
`dense_modal`). What did the numerical damping buy and what did it cost? Which family of schemes damps the high modes **and** stays second order?

In [ ]:
%%fenicsx -np 1
# TODO: Q2(b) — numerically damped Newmark

> *your answer (Q2b)*

**(c) (10)** Slenderness. For $L/H$ = 5, 10, 20, 40 ($H=0.1$, $2L/H\times4$ cells, P2), print $\omega_1$ and $\omega_2$ (bending, by shape) divided by
Euler–Bernoulli. How does the deviation scale with $H/L$? Which physics does Euler–Bernoulli leave out?

In [ ]:
%%fenicsx -np 2
# TODO: Q2(c) — slenderness

> *your answer (Q2c)*

## Q3 — Buckling (30 pts)

The lab's column ($b=0.05$, $h=0.025$, clamped–free, $\lambda$ in units of the first Euler load).

**(a) (10)** Element order and mesh: P1 on $4\times4\times30$ and $8\times8\times60$; P2 on $2\times2\times20$ and $4\times4\times40$. Print $\lambda_1$, $\lambda_2$ (with their axes), the DOFs and the time.
Which is cheaper for 0.5 % accuracy?

In [ ]:
%%fenicsx -np 2
# TODO: Q3(a) — element order and mesh

> *your answer (Q3a)*

**(b) (12)** Slenderness and shear: $L$ = 0.25, 0.5, 1, 2 (P2, $4\times4\times(40L)$ cells, at least 10). For each, print every load factor up to the $x$-2 mode divided by its Euler value, and the
Engesser estimate $1/(1+P_E/(\kappa GA))$ for the $x$-2 mode ($\kappa=5/6$). How does the deviation from Euler scale with $L$?

In [ ]:
%%fenicsx -np 2
# TODO: Q3(b) — slenderness and shear

> *your answer (Q3b)*

**(c) (8)** Which $K_G$? Compare the full form with $\sigma^0_{zz}u_{k,z}v_{k,z}$ (`kg="zz"`) on P2 $4\times4\times40$, mode by mode. Where does the column carry stresses other than $\sigma_{zz}$ in
the pre-buckling state? (Do **not** run `kg="zonly"` — it takes minutes; Q1c tells you what it gives.)

In [ ]:
%%fenicsx -np 2
# TODO: Q3(c) — full K_G vs sigma_zz only

> *your answer (Q3c)*

## Q4 — Debugging an LLM-generated Newmark script (20 pts)

A student asked an LLM for "a self-contained DOLFINx 0.11 script: a 2D cantilever (plane stress, P2) under a suddenly applied tip load, Newmark average
acceleration with 5 % Rayleigh damping in mode 1, print the peak tip deflection over the static one". The script below runs and prints plausible numbers.
It has **four** defects in the time integration.

**(a) (4)** Run it and record the output.

**(b) (8)** List the four defects and the test that exposes each (energy without damping, the oscillator test of `sdof.py`, the logarithmic decrement, …).

**(c) (8)** Correct the script (second cell) and verify: energy conservation with $\zeta=0$, the mode-1 peak against $1+e^{-\zeta\pi/\sqrt{1-\zeta^2}}$, and the damping ratio from the logarithmic
decrement.

In [ ]:
%%fenicsx -np 1
# LLM-generated (as received)
import numpy as np, ufl
from mpi4py import MPI
from petsc4py import PETSc
from slepc4py import SLEPc
from dolfinx import fem, mesh
from dolfinx.fem.petsc import assemble_matrix, assemble_vector, apply_lifting, set_bc

comm = MPI.COMM_WORLD
L, H, E, nu, rho, P, zeta = 1.0, 0.1, 210e9, 0.3, 7800.0, 1000.0, 0.05
msh = mesh.create_rectangle(comm, [[0, 0], [L, H]], [20, 4], cell_type=mesh.CellType.triangle)
V = fem.functionspace(msh, ("Lagrange", 2, (2,)))
mu, lm = E / (2 * (1 + nu)), E * nu / (1 - nu**2)
u, v = ufl.TrialFunction(V), ufl.TestFunction(V)
eps = lambda w: ufl.sym(ufl.grad(w))
sig = lambda w: lm * ufl.tr(eps(w)) * ufl.Identity(2) + 2 * mu * eps(w)
left = mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], 0)); right = np.sort(mesh.locate_entities_boundary(msh, 1, lambda x: np.isclose(x[0], L)))
ds = ufl.Measure("ds", domain=msh, subdomain_data=mesh.meshtags(msh, 1, right, np.ones(len(right), dtype=np.int32)))
bc = fem.dirichletbc(np.zeros(2), fem.locate_dofs_topological(V, 1, left), V)
kf, mf = fem.form(ufl.inner(sig(u), eps(v)) * ufl.dx), fem.form(rho * ufl.inner(u, v) * ufl.dx)
K = assemble_matrix(kf, bcs=[bc], diag=1.0); K.assemble()
M = assemble_matrix(mf, bcs=[bc], diag=1.0); M.assemble()
f = assemble_vector(fem.form(ufl.dot(fem.Constant(msh, np.array([0.0, -P / H])), v) * ds(1)))
apply_lifting(f, [kf], [[bc]]); f.ghostUpdate(PETSc.InsertMode.ADD, PETSc.ScatterMode.REVERSE); set_bc(f, [bc])
g = assemble_vector(fem.form(v[1] / H * ds(1))); set_bc(g, [bc])
def lu(k):
    k.setType("preonly"); k.getPC().setType("lu"); k.getPC().setFactorSolverType("mumps"); return k
# first frequency
M0 = assemble_matrix(mf, bcs=[bc], diag=0.0); M0.assemble()
e = SLEPc.EPS().create(comm); e.setOperators(K, M0); e.setProblemType(SLEPc.EPS.ProblemType.GHEP); e.setDimensions(1)
e.setTarget(0.0); e.setWhichEigenpairs(SLEPc.EPS.Which.TARGET_MAGNITUDE); e.getST().setType("sinvert"); lu(e.getST().getKSP()); e.solve()
w1 = np.sqrt(e.getEigenvalue(0).real); T1 = 2 * np.pi / w1; f1 = w1 / (2 * np.pi)     # first frequency [Hz]
us = f.duplicate(); ks = lu(PETSc.KSP().create(comm)); ks.setOperators(K); ks.solve(f, us); ust = g.dot(us)

def run(zeta, dt, n, beta=0.25, gamma=0.5):
    aM, aK = zeta * f1, zeta / f1                                                # Rayleigh coefficients
    C = M0.copy(); C.scale(aM); C.axpy(aK, K)                                     # Rayleigh damping
    A = M.copy(); A.axpy(gamma * dt, C); A.axpy(beta * dt, K)                     # effective matrix
    ksp = lu(PETSc.KSP().create(comm)); ksp.setOperators(A)
    uu, vv, a, up, vp, r, tmp = (f.duplicate() for _ in range(7))
    uu.set(0); vv.set(0); k0 = lu(PETSc.KSP().create(comm)); k0.setOperators(M); k0.solve(f, a)
    tip, en = [0.0], [0.0]
    for _ in range(n):
        uu.copy(up); up.axpy(dt, vv); up.axpy(0.5 * dt**2, a)                    # predictor
        vv.copy(vp); vp.axpy((1 - gamma) * dt, a)
        f.copy(r); K.mult(up, tmp); r.axpy(-1, tmp); C.mult(vp, tmp); r.axpy(-1, tmp); set_bc(r, [bc], alpha=0.0)
        ksp.solve(r, a)
        up.copy(uu); uu.axpy(beta * dt**2, a)                                     # corrector
        vp.copy(vv); vv.axpy(beta * dt, a)
        tip.append(g.dot(uu))
        K.mult(uu, tmp); M0.mult(vv, r); en.append(0.5 * vv.dot(r) + 0.5 * uu.dot(tmp) - f.dot(uu))
    return np.array(tip), np.array(en)

tip, en = run(zeta, T1 / 100, 1000)
print(f"omega_1 = {w1:.2f} rad/s,  static tip {ust:.4e}")
print(f"5 % damping: peak tip / static = {tip.min() / ust:.4f},  final tip / static = {tip[-1] / ust:.4f}")

In [ ]:
%%fenicsx -np 1
# TODO: Q4(c) — the corrected script and its verification

> *your list of defects and tests (Q4b), and your verification (Q4c)*

## 🤖 AI-usage log

*Required in every assignment and project. Using an LLM is encouraged and never penalized — an empty or vague log is.*

| # | Tool / model | What I asked (key prompt, shortened) | What it got wrong or missed | How I detected it | How I fixed it |
|---|---|---|---|---|---|
| 1 | | | | | |
| 2 | | | | | |
| 3 | | | | | |

**Code provenance** — which cells/functions were mostly generated, which did you write or substantially change?

> …

**Verification** — how do you know the final result is correct (MMS rates, benchmark value, conservation check, parallel = serial)?

> …

**Reflection (2–3 sentences)** — what would you do differently when prompting next time?

> …